3.2 OBIETTIVO
1. Simulare il capitale del giocatore nel tempo usando un approccio Monte Carlo: ad ogni passo
andrà simulato il risultato della singola partita e aggiornato il capitale. Si imposti un numero
massimo di partite (per evitare cicli infiniti) di almeno 50.000 partite.
2. Raccogliere statistiche rilevanti in funzione del tempo T:
a. Stimare le probabilità di rovina e di successo;
b. Si traccino gli andamenti del capitale del giocatore in funzione del tempo;
3. Analizzare le prestazioni
a. Valutare se i risultati ottenuti cambiano significativamente riducendo il numero di
partite massimo;
b. Considerando i soli casi in cui il giocatore si rovina, calcolare il tempo atteso necessario
per la rovina.
c. Considerando i soli casi in cui il giocatore ha successo, calcolare il tempo atteso
necessario per il successo.
4. Verificare la soluzione teorica
a. Ripetendo la simulazione più volte, verificare che la probabilità di rovina converge alla
soluzione teorica

P(rovina) = (1-(p/1-p)^C0)/(1-(p/1-p)^CMax)  se p != 0.5
P(rovina) = 1 - C0/CMax  se p = 0.5
 
b. Ripetere l’esercizio nel caso in cui il giocatore non può raggiungere il successo (cioè
C_max è infinito): provare tutti i casi con p=0.05, 0.15, 0.25, ..., 0.85, 0.95.
c. Verificare che se p>1/2 la rovina non è certa.

5. Effettuare una analisi statistica della probabilità di estinzione (ripetere la simulazione K volte).
a. Stimare attesa e varianza della probabilità di rovina;
b. Per K=50, 100, 200, 2000, fornire un intervallo di confidenza del 95% per la probabilità
di rovina;
c. Calcolare 50 media campionarie usando campioni di 10, 30, 50, 100 prove e tracciarne
le rispettive distribuzioni: mostrare che, specie se il campione è grande, valgono i
risultati del Teorema Limite Centrale.
d. Tracciare un grafico della probabilità di rovina al variare della probabilità p della
distribuzione considerata per X. Tracciare una retta di regressione e valutarne la
significatività.

Modifica il programma per seguire queste istruzioni mantenendo comunque la struttura del codice originale.

I grafici devono essere con plotly express, in modo da essere interattivi ed intuitivi.

Mantieni comunque il codice comprensibile e commentato.



In [37]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import random

def generate_random_number():
    return round(random.random(), 1)

# Funzione per simulare una singola partita
def play_game(p):
    return 1 if random.random() < p else -1
# Funzione per simulare il gioco del giocatore
def simulate_gambler(C0, CMax, p, max_games):
    capital = C0
    history = [capital]
    for _ in range(max_games):
        result = play_game(p)
        capital += result
        history.append(capital)
        if capital <= 0 or capital >= CMax:
            break
    return history, capital >= CMax

# Funzione per calcolare la probabilità teorica di rovina
def theoretical_ruin_prob(C0, CMax, p):
    if p == 0.5:
        return 1 - C0 / CMax
    else:
        q = 1 - p
        return (1 - (p / q) ** C0) / (1 - (p / q) ** CMax)
# Funzione per eseguire la simulazione Monte Carlo
def monte_carlo_simulation(C0, CMax, p, max_games, num_simulations):
    results = []
    for _ in range(num_simulations):
        p = generate_random_number()  # randomizza p ad ogni simulazione
        history, success = simulate_gambler(C0, CMax, p, max_games)
        results.append((history, success))
    return results

# Parametri di simulazione
C0 = 50  # Capitale iniziale
CMax = 5000  # Capitale massimo per il successo
p = generate_random_number()  # Probabilità di vincita
max_games = 50000  # Numero massimo di partite
num_simulations = 1000  # Numero di simulazioni Monte Carlo
# Eseguire la simulazione Monte Carlo
simulation_results = monte_carlo_simulation(C0, CMax, p, max_games, num_simulations)
# Analizzare i risultati
ruin_count = sum(1 for _, success in simulation_results if not success)
success_count = num_simulations - ruin_count
ruin_prob = ruin_count / num_simulations
theoretical_prob = theoretical_ruin_prob(C0, CMax, p)
print(f"Probabilità di rovina stimata: {ruin_prob:.4f}")
print(f"Probabilità di rovina teorica: {theoretical_prob:.4f}")
# Tracciare l'andamento del capitale nel tempo per alcune simulazioni
sample_histories = [history for history, _ in simulation_results[:-1]]
fig = go.Figure()
for i, history in enumerate(sample_histories):
    fig.add_trace(go.Scatter(y=history, mode='lines', name=f'Simulazione {i+1}'))
fig.update_layout(title='Andamento del capitale nel tempo',
                  xaxis_title='Numero di partite',
                  yaxis_title='Capitale')
fig.show()
# Calcolare il tempo atteso per la rovina e il successo
ruin_times = [len(history) for history, success in simulation_results if not success]
success_times = [len(history) for history, success in simulation_results if success]
expected_ruin_time = np.mean(ruin_times) if ruin_times else float('inf')
expected_success_time = np.mean(success_times) if success_times else float('inf')
print(f"Tempo atteso per la rovina: {expected_ruin_time:.2f} partite")
print(f"Tempo atteso per il successo: {expected_success_time:.2f} partite")
# Analisi statistica della probabilità di rovina
K_values = [50, 100, 200, 2000]
for K in K_values:
    sub_simulations = monte_carlo_simulation(C0, CMax, p, max_games, K)
    sub_ruin_count = sum(1 for _, success in sub_simulations if not success)
    sub_ruin_prob = sub_ruin_count / K
    std_error = np.sqrt(sub_ruin_prob * (1 - sub_ruin_prob) / K)
    conf_interval = (sub_ruin_prob - 1.96 * std_error, sub_ruin_prob + 1.96 * std_error)
    print(f"K={K}: Probabilità di rovina stimata: {sub_ruin_prob:.4f}, Intervallo di confidenza 95%: {conf_interval}")
# Distribuzioni delle medie campionarie
sample_sizes = [10, 30, 50, 100]
for size in sample_sizes:
    means = []
    for _ in range(50):
        sample = random.choices([1 if not success else 0 for _, success in simulation_results], k=size)
        means.append(np.mean(sample))
    fig = px.histogram(means, nbins=20, title=f'Distribuzione delle medie campionarie (n={size})')
    fig.show()
# Probabilità di rovina al variare di p
p_values = np.arange(0.05, 1.0, 0.05)
ruin_probs = []
for p_val in p_values:
    sim_results = monte_carlo_simulation(C0, CMax, p_val, max_games, num_simulations)
    ruin_count = sum(1 for _, success in sim_results if not success)
    ruin_probs.append(ruin_count / num_simulations)
df = pd.DataFrame({'p': p_values, 'Ruin_Prob': ruin_probs})
fig = px.scatter(df, x='p', y='Ruin_Prob', title='Probabilità di rovina al variare di p')
fig.add_trace(go.Scatter(x=df['p'], y=np.poly1d(np.polyfit(df['p'], df['Ruin_Prob'], 1))(df['p']),
                         mode='lines', name='Retta di regressione'))
fig.show()
# Regressione e significatività
model = smf.ols('Ruin_Prob ~ p', data=df).fit()
print(model.summary())